# Federated Learning for Heart Disease Prediction
# Flower + PyTorch + FedAvg

# Setup & Imports Libraries

In [1]:
# ============================================================
# Federated Learning for Heart Disease Prediction
# Flower + PyTorch + FedAvg
# ============================================================

import os
import random
import warnings

import numpy as np
import pandas as pdZ

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import flwr as fl

warnings.filterwarnings("ignore")

In [2]:
# ============================================================
# Reproducibility
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device:", DEVICE)

Using Device: cpu


# Define Client Dataset Paths

In [2]:
client_paths = {
    "Hospital_A": "data/clients/hospital_a.csv",
    "Hospital_B": "data/clients/hospital_b.csv",
    "Hospital_C": "data/clients/hospital_c.csv",
    "Hospital_D": "data/clients/hospital_d.csv"
}

In [5]:
clients = {}

for client_name, path in client_paths.items():

    df = pd.read_csv(path)
    clients[client_name] = df

    print(f"{client_name}")
    print(df.shape)
    


Hospital_A
(303, 14)
Hospital_B
(294, 14)
Hospital_C
(123, 14)
Hospital_D
(200, 14)


In [6]:
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,4,140.0,260.0,0.0,1,112.0,1.0,3.0,2.0,0.0,7.0,1
1,44,1,4,130.0,209.0,0.0,1,127.0,0.0,0.0,NaN,0.0,7.0,0
2,60,1,4,132.0,218.0,0.0,1,140.0,1.0,1.5,3.0,0.0,7.0,1
3,55,1,4,142.0,228.0,0.0,1,149.0,1.0,2.5,1.0,0.0,7.0,1
4,66,1,3,110.0,213.0,1.0,2,99.0,1.0,1.3,2.0,0.0,7.0,0


In [9]:
for client_name, df in clients.items():
    print(client_name)
    print(df.isnull().sum())


Hospital_A
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64
Hospital_B
age           0
sex           0
cp            0
trestbps      1
chol         23
fbs           8
restecg       1
thalach       1
exang         1
oldpeak       0
slope       190
ca            0
thal          0
target        0
dtype: int64
Hospital_C
age          0
sex          0
cp           0
trestbps     2
chol         0
fbs         75
restecg      1
thalach      1
exang        1
oldpeak      6
slope       17
ca           0
thal         0
target       0
dtype: int64
Hospital_D
age           0
sex           0
cp            0
trestbps     56
chol          7
fbs           7
restecg       0
thalach      53
exang        53
oldpeak      56
slope       102
ca            0
thal          0
target        0
dtype: int64


In [10]:
df.isnull().sum()

age           0
sex           0
cp            0
trestbps     56
chol          7
fbs           7
restecg       0
thalach      53
exang        53
oldpeak      56
slope       102
ca            0
thal          0
target        0
dtype: int64